# Классификация ботов — quickstart

Ноутбук показывает, как загрузить данные, собрать пару простейших признаков и получить
валидный `submission.csv`. Это **не** решение задачи: скор такого baseline будет чуть выше
константы. Дальше — ваша работа.

Условие и описание метрики — в `README.md`.

In [1]:
import numpy as np
import pandas as pd

train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))
train.head()

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


In [2]:
events.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,ck_5efbea1befdefe1b,2026-04-26 09:11:24,200,item_view,desktop,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1027450.0,elektronika,kaliningrad,pro,NaN,NaN,NaN,NaN
1,ck_c4ca1434f3778f1d,2026-04-20 14:04:31,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,telefony,habarovsk,NaN,iphone 13 128,4.0,NaN,NaN
2,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,WEB,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.0,kvartiry_prodazha,kirov,private,NaN,NaN,675.0,276.0
3,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.0,NaN,NaN
4,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,WEB,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.0,NaN,NaN


## Очистка данных

In [3]:
events['platform'] = events.platform.str.lower()
events = events.sort_values(['cookie_id', 'event_ts']).reset_index(drop=True)

## Осмотреться

Прежде чем считать агрегаты, стоит посмотреть на данные: какие типы событий бывают, что
лежит в `platform` и `user_agent`, где пропуски, всё ли уникально.

In [4]:
print(events.event_name.value_counts(), '\n')
print(events.platform.value_counts(), '\n')
print('пропуски по колонкам:')
print(events.isna().mean().round(3))
print(events.event_name.value_counts())
print(events.groupby('eid').event_name.unique())

event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64 

platform
web        138792
android    126875
desktop     46866
ios         12269
iphone       4103
Name: count, dtype: int64 

пропуски по колонкам:
cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.348
item_category    0.100
item_location    0.072
seller_type      0.407
search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
dtype: float64
event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403

## Окно наблюдения

Признаки считаем только по событиям внутри окна: `window_start_ts <= event_ts < window_end_ts`.
Это требование из условия, а не рекомендация.

In [5]:
def events_in_window(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print(len(ev_tr), len(ev_te))

198436 89690


## Простейшие признаки

Два агрегата — сколько событий и сколько разных объявлений. Этого заведомо мало.

In [6]:
def basic_features(ev, meta):
    g = ev.groupby('cookie_id')
    f = pd.DataFrame({
        'n_events': g.size(),
        'item_nunique': g.item_id.nunique(),
    })
    f = meta[['cookie_id']].merge(f.reset_index(), on='cookie_id', how='left')
    return f.fillna(0)


Xtr = basic_features(ev_tr, train)
Xte = basic_features(ev_te, test)
ytr = train.target.values
Xtr.head()

,cookie_id,n_events,item_nunique
0,ck_54a059eb7d3ea68b,7,4
1,ck_7e4de46eeab82974,41,19
2,ck_9320229ef6304522,36,19
3,ck_30ccd25bc1714ed9,21,10
4,ck_a77c5f05948cdeef,29,16


## Валидация

Тест лежит **позже** трейна по времени, поэтому и валидацию честно делать по времени, а не
случайным сплитом.

Метрику берём из `metric.py` — это ровно тот код, которым считает проверяющая система.
Своя реализация почти наверняка разойдётся с официальной на одинаковых `score`:
их нельзя разделять, группа равных значений отмечается целиком.

In [7]:
from sklearn.ensemble import RandomForestClassifier
from metric import precision_at_recall   # официальная реализация, ей же считает автопроверка

is_valid = train.window_start_ts.ge('2026-04-17').values
cols = ['n_events', 'item_nunique']

model = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0)
model.fit(Xtr.loc[~is_valid, cols], ytr[~is_valid])
p_va = model.predict_proba(Xtr.loc[is_valid, cols])[:, 1]

print('P@R0.7 на валидации:', round(precision_at_recall(ytr[is_valid], p_va), 4))
print('доля ботов (это уровень константы):', round(ytr[is_valid].mean(), 4))

P@R0.7 на валидации: 0.1025
доля ботов (это уровень константы): 0.082


## Сабмит

In [8]:
model.fit(Xtr[cols], ytr)
sub = pd.DataFrame({
    'cookie_id': Xte.cookie_id,
    'score': model.predict_proba(Xte[cols])[:, 1],
})
assert len(sub) == len(test) and sub.score.between(0, 1).all()
sub.to_csv('submission.csv', index=False)
sub.head()

,cookie_id,score
0,ck_315fb710a0e371e7,0.052951
1,ck_a76ee3b3e3e522fd,0.058161
2,ck_94c9a4d382689e82,0.037449
3,ck_8eaf9509ad9462a0,0.058161
4,ck_9a88a5a989cb5bc6,0.034818


## Куда копать дальше

Подсказок по конкретным признакам не будет — это и есть содержание задания. Несколько
вопросов, которые стоит себе задать:

* чем поток событий робота отличается от потока событий человека, если смотреть не на
  количество, а на **моменты времени**;
* что полезного лежит в строке `user_agent` и почему её нельзя брать как есть;
* насколько разнообразно то, что смотрит кука: объявления, категории, запросы, страницы выдачи;
* всё ли в порядке с самим файлом событий — порядок строк, дубликаты, пропуски;
* какие признаки бесполезны, потому что описывают технические характеристики, а не поведение.

Визуализация данных 

In [9]:
bot0_id = train[train.target == 1].cookie_id.iloc[0]
human0_id = train[train.target == 0].cookie_id.iloc[0]
bot1_id = train[train.target == 1].cookie_id.iloc[1]
human1_id = train[train.target == 0].cookie_id.iloc[1]
display(ev_tr[ev_tr.cookie_id == bot0_id].sort_values('event_ts'))
display(ev_tr[ev_tr.cookie_id == bot1_id].sort_values('event_ts'))
display('----------------------------------------------------------------------')
display('ЛЮДИ')
display('----------------------------------------------------------------------')
display(ev_tr[ev_tr.cookie_id == human0_id].sort_values('event_ts'))
display(ev_tr[ev_tr.cookie_id == human1_id].sort_values('event_ts'))


,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,window_start_ts,window_end_ts
210245,ck_e00e1e6c2a2ab4ac,2026-04-06 21:09:49,100,search_results_view,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,NaN,uslugi,sankt-peterburg,NaN,сборка мебели,3.0,NaN,NaN,2026-04-06,2026-04-07
210246,ck_e00e1e6c2a2ab4ac,2026-04-06 21:39:36,100,search_results_view,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,NaN,uslugi,barnaul,NaN,мастер на все руки,5.0,NaN,NaN,2026-04-06,2026-04-07
210247,ck_e00e1e6c2a2ab4ac,2026-04-06 21:39:38,100,search_results_view,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,NaN,zapchasti,sankt-peterburg,NaN,шины зимние,4.0,NaN,NaN,2026-04-06,2026-04-07
210248,ck_e00e1e6c2a2ab4ac,2026-04-06 21:39:45,300,contact_phone_show,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,1024813.0,avtomobili,sankt-peterburg,pro,NaN,NaN,NaN,NaN,2026-04-06,2026-04-07
210249,ck_e00e1e6c2a2ab4ac,2026-04-06 21:40:03,210,photo_swipe,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,1021391.0,kvartiry_prodazha,sankt-peterburg,private,NaN,NaN,NaN,NaN,2026-04-06,2026-04-07
210250,ck_e00e1e6c2a2ab4ac,2026-04-06 21:40:17,100,search_results_view,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,NaN,NaN,sankt-peterburg,NaN,джинсы levis,4.0,NaN,NaN,2026-04-06,2026-04-07
210251,ck_e00e1e6c2a2ab4ac,2026-04-06 21:40:33,400,favorite_add,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,1016533.0,zapchasti,sankt-peterburg,pro,NaN,NaN,NaN,NaN,2026-04-06,2026-04-07
210252,ck_e00e1e6c2a2ab4ac,2026-04-06 21:40:40,200,item_view,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,1032690.0,uslugi,sankt-peterburg,pro,NaN,NaN,NaN,NaN,2026-04-06,2026-04-07
210253,ck_e00e1e6c2a2ab4ac,2026-04-06 23:04:32,200,item_view,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,1055889.0,zapchasti,sankt-peterburg,private,NaN,NaN,NaN,NaN,2026-04-06,2026-04-07
210254,ck_e00e1e6c2a2ab4ac,2026-04-06 23:04:40,100,search_results_view,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,NaN,zapchasti,sankt-peterburg,NaN,шины зимние,2.0,NaN,NaN,2026-04-06,2026-04-07


,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,window_start_ts,window_end_ts
124518,ck_86bb8d44fd720965,2026-04-06 13:17:05,200,item_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,1055299.0,rabota,chelyabinsk,pro,NaN,NaN,97.0,735.0,2026-04-06,2026-04-07
124519,ck_86bb8d44fd720965,2026-04-06 13:17:17,200,item_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,1038387.0,rabota,moskva,private,NaN,NaN,311.0,308.0,2026-04-06,2026-04-07
124520,ck_86bb8d44fd720965,2026-04-06 13:17:25,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,rabota,moskva,NaN,вакансия оператор,1.0,281.0,640.0,2026-04-06,2026-04-07
124521,ck_86bb8d44fd720965,2026-04-06 13:17:53,200,item_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,1037805.0,rabota,kemerovo,private,NaN,NaN,274.0,289.0,2026-04-06,2026-04-07
124522,ck_86bb8d44fd720965,2026-04-06 13:18:38,200,item_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,1008376.0,rabota,kazan,NaN,NaN,NaN,0.0,187.0,2026-04-06,2026-04-07
124523,ck_86bb8d44fd720965,2026-04-06 15:31:50,100,search_results_view,desktop,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,rabota,moskva,NaN,подработка,1.0,383.0,314.0,2026-04-06,2026-04-07
124524,ck_86bb8d44fd720965,2026-04-06 15:32:09,200,item_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,1027002.0,NaN,moskva,private,NaN,NaN,251.0,341.0,2026-04-06,2026-04-07
124525,ck_86bb8d44fd720965,2026-04-06 15:32:34,210,photo_swipe,desktop,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,1008474.0,rabota,ivanovo,private,NaN,NaN,272.0,536.0,2026-04-06,2026-04-07
124526,ck_86bb8d44fd720965,2026-04-06 15:33:08,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,rabota,orenburg,NaN,вакансия продавец,1.0,365.0,124.0,2026-04-06,2026-04-07
124527,ck_86bb8d44fd720965,2026-04-06 15:33:30,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,rabota,cheboksary,NaN,работа водителем,7.0,194.0,360.0,2026-04-06,2026-04-07


'----------------------------------------------------------------------'

'ЛЮДИ'

'----------------------------------------------------------------------'

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,window_start_ts,window_end_ts
78333,ck_54a059eb7d3ea68b,2026-04-06 01:31:27,100,search_results_view,android,Avito/120.0 (Android 13; SM-A515F) okhttp/4.11.0,NaN,kvartiry_prodazha,sankt-peterburg,NaN,купить 1к квартиру,1.0,NaN,NaN,2026-04-06,2026-04-07
78334,ck_54a059eb7d3ea68b,2026-04-06 01:31:28,200,item_view,android,Avito/120.0 (Android 13; SM-A515F) okhttp/4.11.0,1028802.0,NaN,cheboksary,private,NaN,NaN,NaN,NaN,2026-04-06,2026-04-07
78335,ck_54a059eb7d3ea68b,2026-04-06 01:32:02,200,item_view,android,Avito/120.0 (Android 13; SM-A515F) okhttp/4.11.0,1043481.0,kvartiry_prodazha,sankt-peterburg,pro,NaN,NaN,NaN,NaN,2026-04-06,2026-04-07
78336,ck_54a059eb7d3ea68b,2026-04-06 01:32:22,210,photo_swipe,android,Avito/120.0 (Android 13; SM-A515F) okhttp/4.11.0,1045501.0,elektronika,ekaterinburg,pro,NaN,NaN,NaN,NaN,2026-04-06,2026-04-07
78337,ck_54a059eb7d3ea68b,2026-04-06 01:33:46,100,search_results_view,android,Avito/120.0 (Android 13; SM-A515F) okhttp/4.11.0,NaN,kvartiry_prodazha,sankt-peterburg,NaN,квартира с ремонтом,6.0,NaN,NaN,2026-04-06,2026-04-07
78338,ck_54a059eb7d3ea68b,2026-04-06 01:33:52,100,search_results_view,android,Avito/120.0 (Android 13; SM-A515F) okhttp/4.11.0,NaN,kvartiry_prodazha,bryansk,NaN,новостройка,2.0,NaN,NaN,2026-04-06,2026-04-07
78339,ck_54a059eb7d3ea68b,2026-04-06 01:34:14,200,item_view,android,Avito/120.0 (Android 13; SM-A515F) okhttp/4.11.0,1007610.0,zhivotnye,ufa,pro,NaN,NaN,NaN,NaN,2026-04-06,2026-04-07


,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,window_start_ts,window_end_ts
116724,ck_7e4de46eeab82974,2026-04-06 10:50:26,210,photo_swipe,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1042434.0,kvartiry_arenda,vladivostok,private,NaN,NaN,272.0,199.0,2026-04-06,2026-04-07
116725,ck_7e4de46eeab82974,2026-04-06 10:52:12,200,item_view,desktop,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1012113.0,kvartiry_arenda,volgograd,pro,NaN,NaN,845.0,221.0,2026-04-06,2026-04-07
116726,ck_7e4de46eeab82974,2026-04-06 10:53:13,200,item_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1012113.0,kvartiry_arenda,volgograd,NaN,NaN,NaN,1305.0,694.0,2026-04-06,2026-04-07
116727,ck_7e4de46eeab82974,2026-04-06 10:53:19,300,contact_phone_show,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1020676.0,kvartiry_arenda,sankt-peterburg,private,NaN,NaN,1690.0,439.0,2026-04-06,2026-04-07
116728,ck_7e4de46eeab82974,2026-04-06 10:54:07,200,item_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1011414.0,kvartiry_arenda,tula,pro,NaN,NaN,757.0,557.0,2026-04-06,2026-04-07
116729,ck_7e4de46eeab82974,2026-04-06 10:55:24,200,item_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1000931.0,kvartiry_arenda,voronezh,private,NaN,NaN,103.0,654.0,2026-04-06,2026-04-07
116730,ck_7e4de46eeab82974,2026-04-06 10:56:20,200,item_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1045765.0,kvartiry_arenda,vladivostok,pro,NaN,NaN,440.0,409.0,2026-04-06,2026-04-07
116731,ck_7e4de46eeab82974,2026-04-06 10:56:28,200,item_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1042434.0,kvartiry_arenda,vladivostok,private,NaN,NaN,637.0,759.0,2026-04-06,2026-04-07
116732,ck_7e4de46eeab82974,2026-04-06 10:56:30,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,NaN,kvartiry_arenda,tula,NaN,снять квартиру у метро,1.0,803.0,789.0,2026-04-06,2026-04-07
116733,ck_7e4de46eeab82974,2026-04-06 10:57:19,220,seller_page_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1059407.0,kvartiry_arenda,ryazan,NaN,NaN,NaN,2.0,1046.0,2026-04-06,2026-04-07


считаем кандидатов в признаки

In [10]:
ev = ev_tr.sort_values(['cookie_id', 'event_ts']).copy()
ev['gap'] = ev.groupby('cookie_id').event_ts.diff().dt.total_seconds()

f = ev.groupby('cookie_id').agg(
    n_events=('event_ts', 'size'),
    first_ts=('event_ts', 'min'),
    last_ts=('event_ts', 'max'),
    gap_median=('gap', 'median'),
    gap_std=('gap', 'std'),
    gap_max=('gap', 'max'),
    max_page=('search_page', 'max'),
    n_queries=('search_query', 'nunique'),
    n_categories=('item_category', 'nunique'),
    n_locations=('item_location', 'nunique'),
    n_ua=('user_agent', 'nunique'),
).reset_index()
f['duration_min'] = (f.last_ts - f.first_ts).dt.total_seconds() / 60

f = f.merge(train[['cookie_id', 'target']], on='cookie_id')
f.drop(columns=['cookie_id', 'first_ts', 'last_ts']).groupby('target').median().T

target,0,1
n_events,12.000000,20.000000
gap_median,59.000000,25.000000
gap_std,1732.919959,1368.461303
gap_max,5893.000000,6035.000000
max_page,4.000000,7.000000
n_queries,3.000000,4.000000
n_categories,2.000000,2.000000
n_locations,5.000000,8.000000
n_ua,1.000000,1.000000
duration_min,138.775000,151.533333


можно заметить отличия на 
n_events - боты делают больше действий
gap_median - средняя пауза бота в 2.36 раза меньше
max_page - боты листают больше страничек
n_locations - боты просматривают больше локаций

какие действия чаще делают боты

In [11]:
ev_t = ev.merge(train[['cookie_id', 'target']], on='cookie_id')
pd.crosstab(ev_t.target, ev_t.event_name, normalize='index').T.round(3)

target,0,1
event_name,,
contact_chat_open,0.020,0.014
contact_message_sent,0.010,0.008
contact_phone_show,0.037,0.026
favorite_add,0.066,0.027
item_view,0.365,0.436
login,0.021,0.008
photo_swipe,0.123,0.070
search_results_view,0.305,0.352
seller_page_view,0.053,0.059


favorite_add - люди чаще лайкают 
login - чаще логинятся
photo_swipe - больше листают фото

In [13]:
def build_features(ev, meta):
    ev = ev.sort_values(['cookie_id', 'event_ts']).copy()
    ev['gap'] = ev.groupby('cookie_id').event_ts.diff().dt.total_seconds()

    g = ev.groupby('cookie_id')

    f = pd.DataFrame({
        'n_events': g.size(),
        'item_nunique': g.item_id.nunique(),
        'gap_median': g.gap.median(),
        'max_page': g.search_page.max(),
        'n_locations': g.item_location.nunique(),
        'n_queries': g.search_query.nunique(),
    })

    share = pd.crosstab(ev.cookie_id, ev.event_name, normalize='index')
    share.columns = ['share_' + c for c in share.columns]
    f = f.join(share)

    f = meta[['cookie_id']].merge(f.reset_index(), on='cookie_id', how='left')
    return f.fillna(0)


X1tr = build_features(ev_tr, train)
X1te = build_features(ev_te, test)
y1tr = train.target.values
X1tr.head()

is_valid_build = train.window_start_ts.ge('2026-04-17').values
cols_build = cols_build = [c for c in X1tr.columns if c != 'cookie_id']

model_build = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0)
model_build.fit(X1tr.loc[~is_valid_build , cols_build ], y1tr[~is_valid_build])
p_va_build = model_build.predict_proba(X1tr.loc[is_valid_build , cols_build])[:, 1]

print('P@R0.7 на валидации:', round(precision_at_recall(y1tr[is_valid_build], p_va_build), 4))
print('доля ботов (это уровень константы):', round(y1tr[is_valid_build].mean(), 4))

P@R0.7 на валидации: 0.3019
доля ботов (это уровень константы): 0.082


## Эксперименты 

## Журнал экспериментов

Валидация: обучение на окнах до 2026-04-17, проверка на окнах начиная с 2026-04-17. Модель: RandomForest (300 деревьев, min_samples_leaf=3, random_state=0).

| № | Шаг | Признаки | P@R0.7 |
|---|---|---|---|
| 0 | Константа (всем одинаковый score) | — | 0.0820 |
| 1 | Baseline из quickstart | n_events, item_nunique | 0.1025 |
| 2 | + ритм и поиск | + gap_median, max_page, n_locations, n_queries | 0.2258 |
| 3 | + доли типов событий | + share_* по event_name | 0.3019|